# Build the options data (OptionMetrics → `options_daily`)

Rebuilds `blackrock_data/stock_daily/options_daily/` from OptionMetrics on WRDS: one row per stock per trading day with
options volume, open interest, open interest expiring, implied volatility and dealer gamma.

**To use:** set `DATA_ROOT` and the years to build in the first code cell, run all cells, and type your WRDS username and password when asked.
Each month takes about 20 seconds; all of 2015–2025 takes about an hour. Months already saved are skipped, so the notebook can be re-run after an interruption.

**Requirements:** a WRDS account with OptionMetrics (IvyDB US) access, and `pip install wrds pandas duckdb`.

**Sources on WRDS**

| Table | Used for |
|---|---|
| `optionm.opprcdYYYY` | Every option contract per day: volume, open interest, expiry, gamma |
| `optionm.secprdYYYY` | Underlying stock close (for dollar gamma) |
| `optionm.vsurfdYYYY` | Volatility surface (30-day at-the-money implied volatility) |
| `wrdsapps.opcrsphist` | Link from OptionMetrics `secid` to CRSP `permno` (best-quality links, `score = 1`, valid on that date) |
| `crsp.dsf_v2` (via `crsp_daily/`) | The stock universe and daily share volume |

## 0. Settings

In [ ]:
from pathlib import Path

DATA_ROOT = Path("../../blackrock_data")   # the repo's blackrock_data folder
YEARS = range(2015, 2026)                  # years to build
MONTHS = range(1, 13)                      # months to build (e.g. [5] for a quick one-month test)
OVERWRITE = False                          # True = rebuild months that are already saved

In [ ]:
import time
import numpy as np
import pandas as pd
import duckdb
import wrds                                # pip install wrds

OUT_DIR = DATA_ROOT / "stock_daily" / "options_daily"
CRSP_DIR = DATA_ROOT / "stock_daily" / "crsp_daily"
CONTRACT_SIZE = 100                        # shares per contract when OptionMetrics reports none
EXPIRY_WINDOW_DAYS = 7                     # oi_exp_7d counts contracts expiring within this many calendar days

## 1. Connect to WRDS

You are asked for your WRDS username and password; they are not stored in the notebook.

In [ ]:
db = wrds.Connection(wrds_username=input("WRDS username: ").strip())
OPTIONM_TABLES = set(db.list_tables(library="optionm"))

def optionm_table(name, year):
    """OptionMetrics stores large tables by year (e.g. opprcd2024)."""
    table = f"{name}{year}" if f"{name}{year}" in OPTIONM_TABLES else name
    assert table in OPTIONM_TABLES, f"optionm.{name} not found: check your OptionMetrics access on WRDS"
    return f"optionm.{table}"

## 2. Stock universe

The same stocks as the rest of the project: NYSE-listed US common stocks from CRSP. Read from the repo's `crsp_daily/` files if they exist, otherwise pulled from WRDS with the same filters.

In [ ]:
def load_universe(year, month):
    """date, permno and CRSP share volume for every universe stock-day in one month."""
    start = pd.Timestamp(year, month, 1)
    end = start + pd.offsets.MonthEnd(0)
    files = CRSP_DIR / f"year={year}" / "*.parquet"
    if (CRSP_DIR / f"year={year}").exists():
        universe = duckdb.connect().execute(f"""
            SELECT date, permno, daily_volume AS crsp_vol FROM read_parquet('{files}', hive_partitioning=false)
            WHERE date BETWEEN '{start:%Y-%m-%d}' AND '{end:%Y-%m-%d}'
        """).df()
    else:
        universe = db.raw_sql(f"""
            SELECT dlycaldt AS date, permno, dlyvol AS crsp_vol FROM crsp.dsf_v2
            WHERE dlycaldt BETWEEN '{start:%Y-%m-%d}' AND '{end:%Y-%m-%d}'
              AND primaryexch = 'N' AND sharetype = 'NS' AND securitytype = 'EQTY' AND securitysubtype = 'COM'
              AND usincflg = 'Y' AND issuertype = 'CORP' AND conditionaltype = 'RW' AND tradingstatusflg = 'A'
        """)
    universe["date"] = pd.to_datetime(universe["date"])
    universe["permno"] = universe["permno"].astype("int64")
    universe["crsp_vol"] = pd.to_numeric(universe["crsp_vol"], errors="coerce").astype("float64")
    return universe.drop_duplicates(["date", "permno"]).reset_index(drop=True)

## 3. Pull one month from OptionMetrics

The contract-level data is summed **on WRDS** to one row per stock per day, so only a small table is downloaded.

- Volume and open interest are summed separately for calls and puts.
- `oi_exp_today`: open interest in contracts expiring that day. `oi_exp_7d`: expiring within 7 calendar days.
- Gamma is summed as gamma × open interest × contract size, separately for calls and puts.

In [ ]:
def pull_month(year, month, permnos):
    start = pd.Timestamp(year, month, 1).strftime("%Y-%m-%d")
    end = (pd.Timestamp(year, month, 1) + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d")
    opprcd, secprd, vsurfd = (optionm_table(t, year) for t in ("opprcd", "secprd", "vsurfd"))

    link = db.raw_sql(f"""
        SELECT secid, permno, sdate, edate FROM wrdsapps.opcrsphist
        WHERE score = 1 AND permno IN ({",".join(map(str, sorted(set(permnos))))})
          AND sdate <= '{end}' AND edate >= '{start}'
    """, date_cols=["sdate", "edate"])
    secids = ",".join(str(int(s)) for s in link["secid"].unique())

    contracts = db.raw_sql(f"""
        WITH o AS (
            SELECT secid, date, exdate, cp_flag,
                   coalesce(volume, 0) AS volume, coalesce(open_interest, 0) AS oi, coalesce(gamma, 0) AS gamma,
                   CASE WHEN contract_size > 0 THEN contract_size ELSE {CONTRACT_SIZE} END AS cs
            FROM {opprcd}
            WHERE date BETWEEN '{start}' AND '{end}' AND secid IN ({secids})
        )
        SELECT secid, date,
               count(*)                                                                AS n_contracts,
               sum(CASE WHEN cp_flag = 'C' THEN volume ELSE 0 END)                     AS opt_call_vol,
               sum(CASE WHEN cp_flag = 'P' THEN volume ELSE 0 END)                     AS opt_put_vol,
               sum(CASE WHEN cp_flag = 'C' THEN oi ELSE 0 END)                         AS opt_call_oi,
               sum(CASE WHEN cp_flag = 'P' THEN oi ELSE 0 END)                         AS opt_put_oi,
               sum(CASE WHEN exdate = date THEN oi ELSE 0 END)                         AS oi_exp_today,
               sum(CASE WHEN exdate BETWEEN date AND date + {EXPIRY_WINDOW_DAYS} THEN oi ELSE 0 END) AS oi_exp_7d,
               sum(CASE WHEN cp_flag = 'C' THEN gamma * oi * cs ELSE 0 END)            AS call_gamma_shares,
               sum(CASE WHEN cp_flag = 'P' THEN gamma * oi * cs ELSE 0 END)            AS put_gamma_shares
        FROM o GROUP BY secid, date
    """, date_cols=["date"])
    if contracts.empty:
        return None                         # OptionMetrics has no data for this month yet

    prices = db.raw_sql(f"""
        SELECT secid, date, abs(close) AS und_close FROM {secprd}
        WHERE date BETWEEN '{start}' AND '{end}' AND secid IN ({secids})
    """, date_cols=["date"])
    implied_vol = db.raw_sql(f"""
        SELECT secid, date, avg(impl_volatility) AS iv_atm_30d FROM {vsurfd}
        WHERE date BETWEEN '{start}' AND '{end}' AND secid IN ({secids}) AND days = 30 AND abs(delta) = 50
        GROUP BY secid, date
    """, date_cols=["date"])
    return link, contracts, prices, implied_vol

## 4. Build the stock-day table

1. Attach each `secid` to its `permno`, using only links valid on that date. If a stock maps to two `secid`s on a day, the one with more open interest is kept.
2. Put the result on the universe. Stocks with no listed options keep a row with `has_options = 0` and empty values (not zero).
3. Compute the ratio columns.

In [ ]:
COLUMNS = ["date", "permno", "secid", "has_options", "n_contracts", "und_close",
           "opt_call_vol", "opt_put_vol", "opt_total_vol", "opt_to_stock_vol", "pc_vol_ratio",
           "opt_call_oi", "opt_put_oi", "opt_total_oi", "pc_oi_ratio",
           "oi_exp_today", "oi_exp_7d", "oi_exp_7d_share", "iv_atm_30d", "gex_dollar"]

def ratio(a, b):
    a, b = pd.to_numeric(a, errors="coerce").astype("float64"), pd.to_numeric(b, errors="coerce").astype("float64")
    return a / b.where(b > 0)

def build_table(universe, link, contracts, prices, implied_vol):
    s = contracts.merge(prices, on=["secid", "date"], how="left").merge(implied_vol, on=["secid", "date"], how="left")
    s = s.merge(link, on="secid")
    s = s[(s["date"] >= s["sdate"]) & (s["date"] <= s["edate"])].drop(columns=["sdate", "edate"])
    s["permno"] = s["permno"].astype("int64")
    s = (s.assign(_oi=s["opt_call_oi"] + s["opt_put_oi"]).sort_values(["date", "permno", "_oi"], ascending=[True, True, False])
          .drop_duplicates(["date", "permno"]).drop(columns="_oi"))

    df = universe.merge(s, on=["date", "permno"], how="left")
    df["has_options"] = df["n_contracts"].notna().astype("int8")
    df["opt_total_vol"] = df["opt_call_vol"] + df["opt_put_vol"]
    df["opt_total_oi"] = df["opt_call_oi"] + df["opt_put_oi"]
    df["opt_to_stock_vol"] = ratio(df["opt_total_vol"] * CONTRACT_SIZE, df["crsp_vol"])
    df["pc_vol_ratio"] = ratio(df["opt_put_vol"], df["opt_call_vol"])
    df["pc_oi_ratio"] = ratio(df["opt_put_oi"], df["opt_call_oi"])
    df["oi_exp_7d_share"] = ratio(df["oi_exp_7d"], df["opt_total_oi"])
    # Dealer gamma exposure in dollars per 1% move, assuming dealers are long calls and short puts
    df["gex_dollar"] = (df["call_gamma_shares"] - df["put_gamma_shares"]) * df["und_close"] ** 2 * 0.01
    out = df[COLUMNS].astype({"secid": "Int64", "n_contracts": "Int64"})
    return out.sort_values(["date", "permno"]).reset_index(drop=True)

## 5. Build and save every month

In [ ]:
summary = []
for year in YEARS:
    for month in MONTHS:
        path = OUT_DIR / f"year={year}" / f"options_daily_{year}_{month:02d}.parquet"
        if path.exists() and not OVERWRITE:
            continue
        t0 = time.time()
        universe = load_universe(year, month)
        if universe.empty:
            continue
        pulled = pull_month(year, month, universe["permno"])
        if pulled is None:
            print(f"{year}-{month:02d}: no OptionMetrics data yet, skipped")
            continue
        table = build_table(universe, *pulled)
        path.parent.mkdir(parents=True, exist_ok=True)
        table.to_parquet(path, index=False)
        coverage = table["has_options"].mean()
        summary.append({"month": f"{year}-{month:02d}", "rows": len(table), "with_options_pct": round(100 * coverage, 1)})
        print(f"{year}-{month:02d}: {len(table):,} rows, {coverage:.1%} with options, {time.time() - t0:.0f}s")
pd.DataFrame(summary)

## Columns

| Column | Definition | Known by |
|---|---|---|
| `date`, `permno` | Stock-day key (CRSP) | — |
| `secid` | OptionMetrics security ID | — |
| `has_options` | 1 if the stock had any listed option that day | close of *t* |
| `n_contracts` | Number of option series that day | close of *t* |
| `und_close` | Stock close from OptionMetrics | close of *t* |
| `opt_call_vol`, `opt_put_vol`, `opt_total_vol` | Contracts traded | close of *t* |
| `opt_to_stock_vol` | Option volume × 100 ÷ CRSP share volume | close of *t* |
| `pc_vol_ratio` | Put volume ÷ call volume | close of *t* |
| `opt_call_oi`, `opt_put_oi`, `opt_total_oi` | Open interest | close of *t − 1* |
| `pc_oi_ratio` | Put open interest ÷ call open interest | close of *t − 1* |
| `oi_exp_today` | Open interest in contracts expiring on `date` | close of *t − 1* |
| `oi_exp_7d` | Open interest expiring within 7 calendar days | close of *t − 1* |
| `oi_exp_7d_share` | `oi_exp_7d` ÷ `opt_total_oi` | close of *t − 1* |
| `iv_atm_30d` | Average of the call and put 30-day, 50-delta implied volatility | close of *t* |
| `gex_dollar` | Σ gamma × open interest × contract size × price² × 0.01, calls minus puts | close of *t* |

**Timing.** OptionMetrics reports open interest with a one-day lag: open interest on row *t* is the open interest at the close of *t − 1*
(day-to-day changes in open interest line up with the previous day's volume, correlation 0.91, not the same day's, 0.65; checked on May 2024).
So the open-interest columns can be used to forecast day *t*. Every other column is known only after the close of *t* and must be lagged one day,
which the feature template does.

**Coverage.** OptionMetrics on WRDS currently ends 2025-08-29; later months are skipped and fill in when the notebook is re-run after WRDS updates.